# 이전 4시간 구성 · 선택 복습
현재 필수 수업은 [교재 안내](../../README.md)를 따릅니다. 이 파일은 이전 상세 실습 보존본입니다.


# 프롬프트와 구조화된 이미지 판단

**목표:** 관찰 기준과 출력 계약을 명확히 하고 형식과 의미를 따로 검증합니다.

60분 수업: 그림·개념 설명 → 실행 전 예측 → 코드 실행 → 원문·기록 비교. 앞 세 교시는 휴식 10분 포함, 마지막 교시는 종합 평가 포함입니다.

[교재 안내](../../README.md) · [슬라이드 72~81쪽](https://docs.google.com/presentation/d/1IIy5LG5Yue5sJR5DOzmtW6jOuX0DDIZb3LsswTamjcw/edit) · [배포 PDF](../../../슬라이드/강의자료_2일차_v2.pdf#page=72)

**커널:** Python (Day2 LangChain). 설치는 개인 컨테이너 Terminal에서 `bash /workspace/04.LLM_v2/scripts/install_langchain.sh`를 실행합니다. 설치가 끝나면 커널을 선택하고 아래 환경 셀부터 실행하세요. 본문 코드는 실행 가능한 기준 구현이며 마지막에는 별도의 TODO가 있습니다. 코드 작성 여부와 수정할 파일을 TODO 안내에서 확인하세요. TODO는 수업 마지막 10~15분 동안 진행합니다.

In [ ]:
from pathlib import Path
import json, os, sys
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'04.LLM_v2').is_dir()),None)
if ROOT is None:raise RuntimeError('배포 폴더 안에서 노트북을 여세요.')
COURSE=ROOT/'05.VLM_v2'
sys.path.insert(0,str(COURSE))
from vlm_v2.core import *
print('Python:',sys.executable)
print('교재:',COURSE)

## 1. 질문의 범위가 답변을 바꿉니다

넓은 질문은 자유로운 설명을 만듭니다.

대상·근거·모를 때 규칙을 주면 비교하기 쉽습니다.

같은 이미지·모델·출력 상한을 유지합니다.

프롬프트 조건을 한 번에 하나씩 바꿉니다.

그림에서 관찰 범위와 답의 형식이 어떻게 좁아지는지 설명합니다. 답이 짧아졌다고 실제 식별 능력이 좋아졌다고 결론 내리지 않습니다. 동일 원본을 다시 열어 각 주장에 근거가 있는지 봅니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![질문의 범위가 답변을 바꿉니다](../../assets/02_image_questions-01.svg)

## 2. 실습 · 두 질문을 비교합니다

개 사진의 넓은 설명을 요청합니다.

동물과 연결 물체를 지정해 다시 묻습니다.

실제 답변과 지연을 기록합니다.

없는 대상이나 추측이 추가되는지 확인합니다.

같은 모델·이미지에서 프롬프트만 바꿉니다. 필요하면 두 결과를 재실행하되 매번 새 기록으로 구분하세요. 사진을 직접 보고 기대 동물과 연결 물체를 먼저 적습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
vlm=make_model(vision=True,num_predict=384)
path=ROOT/'Data/images/05.VLM/dog.jpg'
from PIL import Image
from IPython.display import display
display(Image.open(path))
questions=['사진을 설명하세요.','동물과 동물에 연결된 물체를 설명하세요. 직접 보이는 근거만 쓰고 불분명하면 unknown이라고 하세요.']
run=new_run('comparison');rows=[]
for q in questions:
    r=observe(vlm,path,structured=False,question=q);rows.append(r)
    append_record(run/'observations.jsonl',r)
    print(r['status'],r['raw'],r.get('seconds'))

## 3. JSON은 다음 코드와의 약속입니다

자유로운 문장은 비교·분기하기 어렵습니다.

label·confidence·evidence 필드를 사용합니다.

허용 label과 자료형을 정합니다.

형식을 맞춘 답도 이미지와 모순될 수 있습니다.

confidence는 모델이 출력한 자기 보고 숫자이며 보정된 정답 확률이 아닙니다. clear/caution/unknown은 교육용 분류입니다. 이 결과로 로봇이나 설비를 직접 제어하는 실습은 여기서 하지 않습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![JSON은 다음 코드와의 약속입니다](../../assets/02_image_questions-03.svg)

## 4. 장면 판단 기준을 먼저 읽습니다

사람이나 차량이 보이면 caution입니다.

없다고 확인할 수 있으면 clear입니다.

가림·흐림으로 판단하기 어려우면 unknown입니다.

정지 이미지에서 움직임은 단정하지 않습니다.

기준을 실행 전에 읽어야 원하는 결과에 맞춰 사후 판단하는 것을 줄일 수 있습니다. 이 분류는 공통 사례의 제조 검사 성능 평가를 대체하지 않습니다. 가상 작업대 실험을 추가하면 그에 맞는 label 정의를 새로 정해야 합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
from vlm_v2.core import SCENE_PROMPT
from vlm_v2.structured import ASSESSMENT_SCHEMA,safe_assessment
print(SCENE_PROMPT)
print(json.dumps(ASSESSMENT_SCHEMA,ensure_ascii=False,indent=2))

## 5. LangChain에서 출력 schema를 전달합니다

model.bind(format=schema)는 Ollama 출력 계약입니다.

같은 모델에 이번 요청 옵션을 붙입니다.

응답 content를 Python으로 다시 검사합니다.

모델의 도구 호출을 요구하는 방식은 아닙니다.

생성 형식 제약은 JSON 문법을 돕지만 출처나 이미지 사실을 검증하지 않습니다. 모델이 schema를 지켰는지, 기준에 맞는 label을 골랐는지, 근거가 사진에 있는지를 각각 봅니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
record=observe(vlm,ROOT/'Data/images/05.VLM/road.png',structured=True)
append_record(run/'observations.jsonl',record)
print(record['status'],record['assessment'],record['error'])
print(record['raw'])

## 6. 잘못된 응답을 직접 만들어 검사합니다

범위 밖 confidence와 잘못된 label을 거부합니다.

필수 키·자료형·근거 길이를 확인합니다.

검사 실패 시 unknown과 오류 이유를 남깁니다.

모델이 판단한 unknown과 오류 fallback을 구분합니다.

정상 응답을 흉내 내는 테스트만으로는 실패 처리를 확인하기 어렵습니다. 경계와 잘못된 값을 직접 넣어 검사가 작동하는지 봅니다. bool은 Python에서 숫자처럼 동작할 수 있으므로 별도 거부합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![잘못된 응답을 직접 만들어 검사합니다](../../assets/02_image_questions-06.svg)

In [ ]:
bad='{"label":"go","confidence":true,"evidence":["추측"]}'
assessment,error=safe_assessment(bad)
assert assessment.label=='unknown' and error
print(assessment.to_dict(),error)

## 7. 형식 성공과 의미 정답을 나눕니다

JSON 파싱 성공 여부를 확인합니다.

label이 정한 기준에 맞는지 확인합니다.

evidence가 실제 이미지에 있는지 확인합니다.

confidence가 높아도 모순이면 오답입니다.

예를 들어 evidence에 차량을 적었는데 label이 clear라면 기준과 모순입니다. 이 모순은 JSON 문법 검사로 해결되지 않습니다. 코드는 형식 계약을, 사람은 의미를 검토합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 8. 비교 평가표를 작성합니다

입력 파일과 모델을 고정했는지 기록합니다.

기대 답은 실행 전에 작성합니다.

실제 답과 근거·지연을 별도 평가합니다.

미평가 값은 None으로 둡니다.

수업에서는 두 질문 비교와 구조화 실제 한 건을 수행합니다. 여러 조건 10~20개 확대는 추가 팀 활동입니다. 단일 도로·개 사진을 제조 성능 데이터로 부르지 않습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
evaluation={'input':str(path),'expected_before_run':'직접 작성',
 'comparison_ids':[r['id'] for r in rows],'structured_id':record['id'],
 'format_ok':record['status']=='ok','label_correct':None,'evidence_visible':None}
(run/'review.json').write_text(json.dumps(evaluation,ensure_ascii=False,indent=2))
print(run/'review.json')

## 9. 오류와 지연도 비교 결과입니다

같은 질문이라도 서버 상태에 따라 지연이 달라집니다.

출력 상한·모델 적재·생성 시간을 구분합니다.

오류 원문을 남겨 실패 단계부터 확인합니다.

성공 결과만 모으면 비교가 왜곡될 수 있습니다.

공용 Ollama에 여러 수강생이 동시에 요청하면 대기 시간이 길어집니다. 한 프로그램에서 무한 병렬 호출을 보내지 않습니다. 수업은 비교 조건을 적은 뒤 필요한 요청만 순서대로 보냅니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 10. 2교시 정리 · 카메라에도 같은 계약을 씁니다

이미지 질문에 대상·근거·보류를 넣었습니다.

구조화 결과를 코드와 사람이 나누어 확인했습니다.

다음 교시에는 선택한 프레임마다 같은 검사를 합니다.

현재 사진과 이전 사진을 혼동하지 않습니다.

이론 20분, 실습 25분, 비교 5분, 휴식 10분입니다. 추가 실습은 “사람이 가려져 있으면 clear로 볼 수 있는가?”처럼 기준의 경계를 토론합니다. 프레임 단위 판단과 시간 변화 판단은 다음 교시에서 구분합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 마지막 TODO — 구조화 응답의 허용값 검사

**코드 작성: 있음 · 15분 · 수업 시간 안에 진행**

수정 파일: [practice/assessment_check.py](../../practice/assessment_check.py). JupyterLab 편집기에서 함수를 완성하고 저장한 뒤 아래 로딩 셀과 검사 셀을 다시 실행하세요. 노트북은 학생 파일을 덮어쓰지 않습니다.

**입력·출력 계약:** value가 정확히 label/confidence/evidence 키를 가진 dict인지 검사해 bool을 반환하세요. label은 clear/caution/unknown 문자열, confidence는 bool이 아닌 유한한 0~1 숫자, evidence는 공백 제거 후 1~120자 문자열 1~3개인 list여야 합니다. 잘못된 입력은 예외가 아니라 False입니다. 의미의 정답 여부는 검사하지 않습니다.

**힌트:** dict·키 확인 → label → confidence → evidence 순서로 검사합니다. bool과 NaN/무한대도 확인하세요.

미완성이면 마지막 검사에서 NotImplementedError가 나는 것이 정상입니다. 검사 통과 후 아래 적용 예제의 중간값을 설명하세요. 이 예제는 학생 함수를 직접 사용하며 본문의 기준 클래스/카메라 프로그램을 자동 교체하지 않습니다.

In [ ]:
import importlib.util, inspect, os
runner_path=COURSE/'practice/runner.py'
spec=importlib.util.spec_from_file_location('day2_todo_runner',runner_path)
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
VARIANT=os.environ.get('DAY2_TODO_VARIANT','exercise')
task=runner.load('assessment_check',VARIANT)
print('선택 구현:',VARIANT)
print(inspect.getsource(task.is_valid_assessment))

In [ ]:
print(runner.check('assessment_check',task))

samples=[{'label':'caution','confidence':0.8,'evidence':['차량이 보임']},
         {'label':'clear','confidence':True,'evidence':['차량이 보임']},
         {'label':'clear','confidence':0.95,'evidence':['차량이 보임']}]
for sample in samples:print(task.is_valid_assessment(sample),sample)
print('마지막 예제는 형식이 맞지만 label과 근거가 모순입니다. 사람이 따로 평가하세요.')

### TODO 완료와 참고 정답

- [ ] 함수를 직접 작성하고 검사 셀을 통과했습니다.
- [ ] 경계 입력 한 개를 추가해 예상 결과를 설명했습니다.
- [ ] 적용 예제의 입력과 반환값을 설명했습니다.

강사가 정한 직접 시도 시간 뒤 [참고 정답](../../practice/solutions/assessment_check.py)을 비교하세요. 참고 정답을 실행했다고 학생 파일이 완성되는 것은 아닙니다.

**해설:** 형식이 잘못되면 후속 코드가 값을 신뢰할 수 없습니다. 반대로 형식을 통과한 clear 응답이 차량을 근거로 쓰는 모순은 이 검사만으로 판정하지 않습니다.

추가 모델 요청 없이 함수와 데이터 흐름을 검사합니다. 실제 모델·카메라 결과 평가는 본문 실습에서 수행합니다.

## 완료 확인

실제 입력과 출력 기록을 확인하고, 형식·근거·지연을 구분해 설명합니다. 오류는 숨기지 않고 원문과 함께 남깁니다. 추가 실험은 한 조건만 바꿔 비교합니다.